# Example 11.4.3 / 3: Reading Termination

Companion notebook for **11.4.3 Local and Global Nonlinear Solvers** in the AMPL lab Documentation, section "Reading Termination". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Reading Termination

For a global solve, read both the best feasible objective and the valid global bound. Stopping on a time limit with a remaining gap is not the same as proving optimality. For a local solve, note the start, feasibility residuals, and whether the solver reports local optimality, acceptable convergence, or failure.

A local termination that is not convergence: limiting Ipopt to two iterations with
`ipopt_options` set to `max_iter=2` stops at $x\approx1.0254$, which is not a stationary point.
`solve_result` is `limit` and `solve_result_num` is `400`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
var x >= -2, <= 2, := 1.5;  # starts in the right-hand well

minimize Wells: (x^2 - 1)^2 + 0.1 * x;  # nonconvex: two local minima

Writing model.mod


### amplpy: run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# run.py
ampl.option["ipopt_options"] = "max_iter=2"
ampl.solve(solver="ipopt", verbose=False)
print(ampl.solve_result, ampl.get_value("solve_result_num"))
print(ampl.get_value("solve_message").strip())
print(round(ampl.var["x"].value(), 4))

limit 400
Ipopt 3.12.13: Maximum Number of Iterations Exceeded.
1.0254
